In [ ]:
import os
from time import perf_counter

import numpy as np
import tensorflow as tf
from bayesflow.amortizers import AmortizedPosterior
from bayesflow.networks import InvertibleNetwork
from bayesflow.simulation import GenerativeModel, Prior, Simulator
from bayesflow.trainers import Trainer
from tensorflow.keras.layers import Dense, GRU, LSTM

In [ ]:
from Simulator.simulator import simulator, param_names

In [ ]:
n_params = len(param_names)

In [ ]:
def prior(batch_size: int) -> np.ndarray:
    param_batch = np.zeros((batch_size, 12))
    # alpha
    param_batch[:, 0] = np.random.uniform(0, 0.1, batch_size)
    # beta
    param_batch[:, 1] = np.random.uniform(0, 10, batch_size)
    # delta
    param_batch[:, 2] = np.random.uniform(-3, 3, batch_size)
    # mu_inf
    param_batch[:, 3:8] = np.exp(np.random.normal(0, 1, (batch_size, 5)))
    # mu_susc
    param_batch[:, 8:10] = np.exp(np.random.normal(0, 1, (batch_size, 2)))
    # mu_protect
    param_batch[:, 10:] = np.exp(np.random.normal(0, 1, (batch_size, 2)))
    return param_batch

np.random.seed(42)
_prior_draws = prior(1000)
assert _prior_draws.shape == (1000, n_params)
prior_mean = _prior_draws.mean(axis=0)
prior_std = _prior_draws.std(axis=0)
print(prior_mean, prior_std)

In [ ]:
_sim = simulator(prior(1).flatten())
n_households = _sim.shape[0]
length_time_series = _sim.shape[1]
print(n_households)

In [ ]:
bayesflow_prior = Prior(batch_prior_fun=prior,
                        param_names=param_names)
bayes_simulator = Simulator(simulator_fun=simulator)
generative_model = GenerativeModel(prior=bayesflow_prior, simulator=bayes_simulator,
                                   skip_test=True,
                                   name="Normalizing Flow Generative Model")

In [ ]:
def configurator(forward_dict: dict) -> dict:
    out_dict = {}

    # Extract data (already normalized)
    x = forward_dict["sim_data"].astype(np.float32)    
    idx_keep = np.all(np.isfinite(x), axis=(1, 2, 3))
    out_dict['summary_conditions'] = x[idx_keep]
    
    # if simulations contains nan
    if not np.all(idx_keep):
        print(f'Invalid value(s) encountered...removing {idx_keep.size - np.sum(idx_keep)} entry(ies) from batch')
    
    # Extract params
    if 'parameters' in forward_dict.keys():
        forward_dict["prior_draws"] = forward_dict["parameters"]
    if 'prior_draws' in forward_dict.keys():
        params = forward_dict["prior_draws"].astype(np.float32)
        params = (params - prior_mean) / prior_std
        out_dict['parameters'] = params[idx_keep]
    return out_dict

In [ ]:
class GroupSummaryNetwork(tf.keras.Model):
    
    def __init__(
        self, summary_dim=10, rnn_units=128, n_groups=128, use_lstm=False, **kwargs
    ):
        super().__init__(**kwargs)

        self.rnn = LSTM(rnn_units) if use_lstm else GRU(rnn_units)
        self.out_layer = Dense(summary_dim, activation="linear")
        self.summary_dim = summary_dim
        self.n_groups = n_groups
        
    def call(self, x, **kwargs):
        """Performs a forward pass through the network by first passing `x` through the same rnn network for
        each household and then pooling the outputs across households.

        Parameters
        ----------
        x : tf.Tensor
            Input of shape (batch_size, n_groups, n_time_steps, n_features)

        Returns
        -------
        out : tf.Tensor
            Output of shape (batch_size, summary_dim)
        """
        # iterate over groups
        out_list = []  # list to store outputs of LSTM for each group
        for i in range(self.n_groups):
            out = self.rnn(x[:, i], **kwargs)  # (batch_size, lstm_units)
            out_list.append(out)
        # one could apply a time-resolved equivariant layer here and then pool
        # max pooling over groups
        out = tf.reduce_max(out_list, axis=0)  # (batch_size, lstm_units)
        # apply dense layer
        out = self.out_layer(out, **kwargs)  # (batch_size, summary_dim)
        return out

In [ ]:
power_k_hidden_units = int(np.ceil(np.log2(length_time_series)))
summary_net = GroupSummaryNetwork(summary_dim=n_params*2,
                                  rnn_units=2**power_k_hidden_units,
                                  n_groups=n_households)
inference_net = InvertibleNetwork(num_params=n_params,
                                  num_coupling_layers=6,
                                  coupling_design='spline',
                                  coupling_settings={
                                      "num_dense": 2,
                                      "dense_args": dict(
                                          activation='swish',
                                          kernel_regularizer=tf.keras.regularizers.l2(1e-4)
                                      )
                                  })

In [ ]:
amortizer = AmortizedPosterior(inference_net=inference_net, summary_net=summary_net)
checkpoint_path = 'amortizer-test'

# build the trainer with networks and generative model
trainer = Trainer(amortizer=amortizer,
                  configurator=configurator,
                  generative_model=generative_model,
                  checkpoint_path=checkpoint_path,
                  skip_checks=True,
                  max_to_keep=1)

In [ ]:
presimulate = True
train_network = False

batch_size = 64
iterations_per_epoch = 10
max_epochs = 1000
job_id = int(os.environ.get('SLURM_ARRAY_TASK_ID', 0))

from time import sleep
sleep(job_id)

# np.random.seed(42)
# # check if the data exists
# if os.path.exists('valid_data.npy') and os.path.exists('test_data.npy'):
#     valid_data = np.load('valid_data.npy')
#     test_data = np.load('test_data.npy')
# else:
#     valid_data = generative_model(128)
#     test_data = generative_model(128)
# 
#     # save the data
#     np.save('valid_data.npy', valid_data)
#     np.save('test_data.npy', test_data)

In [ ]:
if presimulate:
    start_time = perf_counter()
    generative_model.presimulate_and_save(batch_size=64, 
                                          folder_path='presimulations',
                                          iterations_per_epoch=iterations_per_epoch,
                                          epochs=1,
                                          extend_from=job_id,
                                          disable_user_input=True)
    end_time = perf_counter()
    print(f'simulation time: {(end_time-start_time)/60} minutes')

In [ ]:
if train_network:
    # simulation done before, start training now
    start_time = perf_counter()
    trainer._setup_optimizer(optimizer=None,
                             epochs=max_epochs,
                             iterations_per_epoch=iterations_per_epoch)

    history = trainer.train_from_presimulation(presimulation_path='presimulations',
                                               optimizer=trainer.optimizer,
                                               max_epochs=max_epochs,
                                               early_stopping=True,
                                               validation_sims=valid_data)

    end_time = perf_counter()
    print(f'training time: {(end_time-start_time)/60} minutes')
else:
    trainer.load_pretrained_network()